# Runner FULL: una combinazione alla volta

Questo notebook coordina i moduli del laboratorio, senza controllare Docker né preparare CSV. Seleziona **un database e una dimensione** in `config/benchmark.yaml`. Gli script host devono rifiutare altri DB attivi; con Compose diretto arrestali esplicitamente. Il runner controlla anche gli endpoint di rete, ma non può inventariare container senza Docker socket.

Il percorso manuale dei notebook 02_ingestion/03_query_benchmarks resta disponibile separatamente: non alternarlo al runner sulla stessa tabella durante una campagna. Il runner confronta gli snapshot della propria campagna, senza usare i percorsi legacy `comparison.*`.
Le nove combinazioni concluse si consultano nel notebook 06; non vanno ripetute per la presentazione. `ESEGUI=False` mantiene inattivo il runner.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import load_config
from src.runner import run_full, campaign_status, unlock_campaign
config = load_config(ROOT / 'config' / 'benchmark.yaml')
print(f'Database: {config.database} | record: {config.dataset_size:,} | batch: {config.batch_size} | warm-up: {config.warmup} | misure: {config.repetitions}')
from src.runner_ui import RunnerProgress, show_summary


Database: mongo | record: 100,000 | batch: 1000 | warm-up: 3 | misure: 20


## Selezione e avvio espliciti

Usa la stessa directory CAMPAGNA per i tre motori e le tre dimensioni. Un protocollo immutabile conserva filtri, seed, warm-up, ripetizioni, batch e checksum dei moduli. Se cambiano, crea una nuova campagna. Ogni combinazione conserva destinazione e checksum del CSV; il runner non itera sulla matrice.

`ESEGUI=False` rende sicuro eseguire tutte le celle per ispezionare la configurazione. Per iniziare imposta `ESEGUI=True` dopo aver controllato DB e destinazione. I dati esistenti vengono rifiutati; `ingestion.confirm_reset: true` autorizza un reset **solo quando la fase ingestione deve essere eseguita**. Riporta sempre il flag a false dopo l'uso.

Le dimensioni 1M/5,8M richiedono autorizzazione separata e `ABILITA_DIMENSIONI_MAGGIORI=True`: abilitarlo solo per una nuova campagna autorizzata. Il timeout SQL/socket resta 120 s, quello dei lock SQL 10 s; un audit integrale potrebbe superarli. Nessun aumento o retry generico dei timeout è automatico.
### Come leggere l'avanzamento
Un pannello aggiornato circa ogni secondo mostra motore, dimensione, campagna e le quattro fasi. `pending` = in attesa; `running` = in corso; `completed` = fase validata e pubblicata; `resumed` = misure precedenti riutilizzate dopo i controlli; `skipped` = fase non eseguita in questa invocazione per una pausa richiesta; `failed` = errore, consultare la diagnostica. I peer `pending` nel riepilogo non indicano un errore del motore corrente.

Durante l'ingestione i contatori sono letti da un osservatore separato: record confermati, batch e percentuale sono reali; la velocità live è approssimata nell'intervallo UI e non sostituisce il throughput salvato. Non viene stimato un tempo residuo. Il 100% dei record non implica che audit e snapshot siano già terminati: attendere `completed` della fase.

Per le query sono indicati scenario, warm-up e misure completate; per gli indici nome e transizioni reali. Le transizioni brevi rimangono espandibili. Avvisi Docker/ambiente sono non bloccanti e restano negli artefatti. La UI non modifica i timer; il suo rendering concorrente può comunque aggiungere carico alla macchina. Per uso headless omettere `progress`.

Il riepilogo finale legge soltanto gli artefatti, anche con `ESEGUI=False`, senza riprendere campagne concluse. I controlli di impronta del codice rimangono attivi: un aggiornamento dei sorgenti non autorizza automaticamente una ripresa di campagne create con codice diverso.


In [2]:
CAMPAGNA = ROOT / 'results' / 'full-campaign-100000'
ESEGUI = False
RIPRENDI = False
FERMATI_DOPO = None  # oppure 'ingestion', 'baseline', 'indexes'
ABILITA_DIMENSIONI_MAGGIORI = False
if ESEGUI:
    try:
        with RunnerProgress(config.database, config.dataset_size, CAMPAGNA) as avanzamento:
            stato = run_full(config, CAMPAGNA, resume=RIPRENDI,
                             stop_after=FERMATI_DOPO,
                             allow_large_dataset=ABILITA_DIMENSIONI_MAGGIORI,
                             progress=avanzamento)
    finally:
        show_summary(CAMPAGNA, config.database, config.dataset_size, results_dir=config.results_dir)
else:
    print('Nessun esperimento avviato. Il riepilogo della campagna si consulta nella cella seguente.')


fase,stato
ingestion,completed
baseline,completed
indexes,completed
indexed,completed


## Checkpoint, errori e ripresa

Le fasi sono ingestione, baseline Q1-Q4, costruzione indici, Q2/Q3 indicizzate. Ogni tentativo ha una directory unica con manifest, raw, riferimenti, versioni, piani, correttezza e summary solo se completato. I raw vengono pubblicati dopo ogni osservazione, fuori dal timer. Le scritture del filesystem possono influenzare le pause fra query; timer, warm-up e statistiche restano quelli comuni.

- Dopo una pausa controllata o un errore, imposta `RIPRENDI=True`: le fasi completate vengono conservate, quelle query incomplete ripartono con nuovi warm-up in un nuovo tentativo, senza mescolare misure.
- Ingestione parziale: nessun append automatico e nessun throughput ricostruito. Occorre un reset esplicito oppure lasciare invariati i dati e fermarsi. Anche un caricamento completo il cui audit è fallito non viene dichiarato valido.
- Costruzione indici parziale: verificare lo stato; `benchmark.confirm_drop_indexes: true` autorizza rimozione dei soli indici del laboratorio e nuova misura della costruzione. I tentativi falliti restano conservati. Non rimuovere indici appartenenti a fasi già completate.
- Dopo un kill/OOM può rimanere `runner.lock`. Arresta/verifica il vecchio kernel prima di chiamare consapevolmente `unlock_campaign(CAMPAGNA, confirm_no_runner=True)`. Il runner non sblocca automaticamente un processo forse ancora vivo.
- Se il manifest di una fase era completato ma mancava solo il checkpoint, la ripresa lo adotta dopo un nuovo audit. Una ripresa già completata verifica i dati senza ricaricarli né ripetere benchmark.

Non cancellare `experiments/` per forzare la ripresa: perderesti il legame con i tentativi. Per un nuovo esperimento usa una nuova campagna e decidi esplicitamente la gestione dei dati persistenti.

In [3]:
show_summary(CAMPAGNA, config.database, config.dataset_size, results_dir=config.results_dir)


## Cambio database e lettura dei risultati

Arresta il database corrente dall'host, poi avvia il successivo, cambia `database` nel YAML e riesegui la configurazione. Per ogni dimensione i primi confronti fra motori sono `pending`; quando arrivano gli snapshot completati, vengono verificati CSV, ID, NULL, Q1 completo e Q2-Q4 con la tolleranza comune. Il riepilogo distingue combinazioni completate, in attesa o fallite.

`protocol.json` e `experiments/<database>_<dimensione>.json` contengono protocollo, checkpoint, errori e percorsi dei tentativi. `status.json` è un riepilogo persistente, mentre `campaign_status()` rilegge gli stati. Questi file non sostituiscono manifest e raw dei singoli run. Nessuna analisi grafica o modalità DEMO viene eseguita qui.